# Practice 4 — обучение BC на Google Colab (GPU)

Датасеты вы собираете локально в Docker (`dataset/train_10k` и `dataset/eval`), а обучение BC-моделей выполняете в Colab с GPU. Доступность GPU зависит от текущих лимитов Colab.

**Перед запуском:** меню `Runtime → Change runtime type → Hardware accelerator → GPU`.

Порядок: проверить GPU → склонировать проект → поставить зависимости → подложить LeRobotDataset → обучить `image`, `both+mlp`, `proprio` и `both+identity` → посмотреть TensorBoard → прогнать rollout → скачать чекпоинты.

In [ ]:
!nvidia-smi

## 1. Клонируем проект

In [ ]:
!git clone https://github.com/Yandex-Practicum/ap-physiclaai-4.git
%cd ap-physiclaai-4

## 2. Зависимости

Устанавливаем закреплённые в `requirements.txt` версии зависимостей проекта. Они согласованы с Docker-образами практики. После установки перезапустите runtime, если Colab попросит это сделать.

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch, torchvision, torchcodec
print(torch.__version__, torchvision.__version__, torchcodec.__version__)
print('CUDA:', torch.cuda.is_available())

## 3. Датасеты

Нужны папки `dataset/train_10k` и `dataset/eval` в формате LeRobotDataset v3 (`meta/`, `data/`, `videos/`). Обе должны содержать `observation.images.front` и `observation.proprio`.

**Вариант А — через Google Drive (рекомендуется).** Локально заархивируйте папку `dataset/` и загрузите архив в Drive (например, в `MyDrive/practice4/`). Затем выполните ячейку ниже и распакуйте.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Подставьте свой путь. Пример, если вы загрузили dataset.zip:
# !unzip -q /content/drive/MyDrive/practice4/dataset.zip -d .
# Или копирование готовой папки dataset/:
!mkdir -p dataset
!cp -r "/content/drive/MyDrive/practice4/dataset/." dataset/
from pathlib import Path
for name in ('train_10k', 'eval'):
    print(name, 'meta:', (Path('dataset') / name / 'meta' / 'info.json').is_file())

**Вариант Б — готовые артефакты курса** (если преподаватель выдал рабочий `ARTIFACTS_URL`):

In [ ]:
# %env ARTIFACTS_URL=https://storage.googleapis.com/<ваш-бакет>/artifacts
# !python3 scripts/download_artifacts.py --datasets train_10k eval

## 4. Обучение четырёх конфигураций

Перед запуском завершите все TODO Урока 5 в `model.py` и `train_bc.py`.

Чекпоинты сохраняются в `logs/<exp>/checkpoints/{best,last}.pt`, логи TensorBoard — в `logs/<exp>/`.

In [ ]:
!python3 train_bc.py --train_dir dataset/train_10k --eval_dir dataset/eval --exp_name bc_image --obs_mode image --proprio_encoder_type mlp --epochs 100 --batch_size 64 --lr 1e-4

In [ ]:
!python3 train_bc.py --train_dir dataset/train_10k --eval_dir dataset/eval --exp_name bc_both_mlp --obs_mode both --proprio_encoder_type mlp --epochs 100 --batch_size 64 --lr 1e-4

In [ ]:
!python3 train_bc.py --train_dir dataset/train_10k --eval_dir dataset/eval --exp_name bc_proprio --obs_mode proprio --proprio_encoder_type mlp --epochs 100 --batch_size 64 --lr 1e-4

In [ ]:
!python3 train_bc.py --train_dir dataset/train_10k --eval_dir dataset/eval --exp_name bc_both_identity --obs_mode both --proprio_encoder_type identity --epochs 100 --batch_size 64 --lr 1e-4

## 5. TensorBoard

Сравните `train/loss` и `eval/loss` для всех четырёх экспериментов.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 6. Rollout-оценка (Success Rate)

Перед запуском завершите все TODO Урока 6 в `inference.py`.

MuJoCo на Colab работает без окна через EGL — задаём `MUJOCO_GL=egl`.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_image/checkpoints/best.pt         --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_both_mlp/checkpoints/best.pt      --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_proprio/checkpoints/best.pt       --model bc --episodes 50 --seed 999
!MUJOCO_GL=egl python3 inference.py --checkpoint logs/bc_both_identity/checkpoints/best.pt --model bc --episodes 50 --seed 999

## 7. Скачать обученные чекпоинты

Архивируем результаты четырёх экспериментов (веса + логи) и скачиваем на свой компьютер. Дальше их можно положить в локальный проект в те же папки и запускать `inference.py`.

In [ ]:
import shutil
from google.colab import files
for n in ['bc_image', 'bc_both_mlp', 'bc_proprio', 'bc_both_identity']:
    shutil.make_archive(n, 'zip', 'logs', n)
    files.download(f'{n}.zip')